# NSDUH 2021–2024 — Data Cleaning Pipeline
**CSE 3104 Final Project** · Caitlyn Bui, Chloe Doan, Erica Kang

This notebook turns the raw concatenated NSDUH public-use file (**232,441 respondents × 2,638 variables**) into a small, analysis-ready table that answers the proposal's questions:

| Proposal question | What the cleaned data provides |
|---|---|
| Did depression rise, fall, or level off 2021→2024? | `mde_py` / `mde_lt` (past-year / lifetime major depressive episode) for every year |
| Do changes differ by age, sex, race/ethnicity, college enrollment? | `age_group`, `sex`, `race_eth`, `race_eth5`, `college_status`, `college_ft_vs_other` |
| Psychological distress (adults) | `k6_pm`, `k6_py_max`, `spd_pm`, `spd_py` (Kessler K6, ages 18+) |
| Mental health treatment received | `mh_tx_py`, `mde_tx` |
| Valid national estimates | survey weights `wt_1yr`…`wt_4yr`, strata `strata`, replicate/PSU `psu` |

**Pipeline stages**

1. Read **metadata only** (names + labels) — no need to load 2,638 columns
2. **Select** ~25 variables, resolving name differences automatically
3. **Load** only those columns
4. **Inspect** the codes each variable uses
5. **Recode missing-value codes** using SAMHSA's own Stata syntax file
6. **Derive** analysis variables (age group, race/ethnicity, college status, unified MDE…)
7. **Validate** with automated checks
8. **Save** a Parquet file + data dictionary

Each step has a short *what / why* note so the reasoning can go straight into the report.

## 0. Setup
All paths live in one place so the notebook can be re-run end to end (the "automated" part of the pipeline). Put the Stata syntax file (`.do`) from SAMHSA in the same `data/` folder as the `.dta`.

In [3]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

DATA_DIR = Path("data")
DTA_PATH = DATA_DIR / "NSDUH_2021_2024.dta"
DO_PATH  = DATA_DIR / "NSDUH_2021_2024-data-stata-supplemental-syntax.do"
OUT_DIR  = DATA_DIR / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_YEARS = [2021, 2022, 2023, 2024]

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.3f}".format)

AttributeError: 'str' object has no attribute 'mkdir'

## 1. Read metadata only

**What:** read just the column names, variable labels, and value labels.

**Why:** the full file is 232,441 × 2,638 ≈ 613 million cells (~4.9 GB as float64). We only need ~25 columns, so loading everything wastes memory and time. `metadataonly=True` returns in seconds and also gives us the **value labels** stored in the `.dta`, which we use throughout to confirm what each code means instead of guessing.

**Column-name case:** the file mixes cases (`year`, `verep`, `irmarit` vs `AGE3`, `NEWRACE2`). SAMHSA's `.do` file starts with `rename *, upper`, so we mirror that and refer to every variable in UPPERCASE.

In [2]:
_, meta = pyreadstat.read_dta(str(DTA_PATH), metadataonly=True)

UPPER_TO_ORIG = {c.upper(): c for c in meta.column_names}
VAR_LABELS    = {c.upper(): lbl for c, lbl in meta.column_names_to_labels.items()}
VALUE_LABELS  = {c.upper(): lbls for c, lbls in meta.variable_value_labels.items()}

print(f"{meta.number_rows:,} rows x {len(meta.column_names):,} columns")

PyreadstatError: File data/NSDUH_2021_2024.dta does not exist!

In [ ]:
def search_vars(pattern):
    # Find variables whose NAME or LABEL matches a regex — handy for checking codebook names.
    rx = re.compile(pattern, re.I)
    hits = [(v, VAR_LABELS.get(v, "")) for v in UPPER_TO_ORIG
            if rx.search(v) or rx.search(VAR_LABELS.get(v, "") or "")]
    return pd.DataFrame(hits, columns=["variable", "label"])

def value_labels(var):
    return VALUE_LABELS.get(var.upper(), {})

# Example: everything related to depression / distress / college
search_vars(r"MDE|K6|SPD|COLLENR")

## 2. Select variables

**What:** map each *concept* in the proposal to one or more candidate NSDUH variable names, and keep the first name that exists in the file.

**Why:** NSDUH occasionally renames variables between releases (e.g. strata `VESTR` → `VESTR_C` in concatenated files). Listing candidates makes the pipeline robust and tells us immediately if a required variable is missing, rather than failing later.

| Concept | NSDUH variable | Universe | Notes |
|---|---|---|---|
| Respondent ID | `QUESTID2` | all | if absent, we create one |
| Survey year | `YEAR` | all | |
| Age category | `AGE3` | all | 1–3 = 12–17, 4–6 = 18–25 |
| Sex | `IRSEX` | all | imputed, no missing |
| Race/ethnicity | `NEWRACE2` | all | 7 categories |
| College enrollment | `COLLENRLST` | **ages 18–22 only** | SAMHSA's own recode; uses exact age internally |
| Youth MDE (lifetime / past year) | `YMDELT`, `YMDEYR` | 12–17 | adolescent module |
| Adult MDE (lifetime / past year) | `AMDELT`, `AMDEYR` | 18+ | adult module |
| K6 distress score | `K6SCMON`, `K6SCMAX` | 18+ | 0–24 |
| Serious psychological distress | `SPDMON`, `SPDYR` | 18+ | K6 ≥ 13 |
| Mental health treatment | `MHTRTPY` (or older `AMHTXRC*`) | varies | treatment questions were redesigned in 2021+ |
| Treatment for depression (if MDE) | `YMDETXRX`, `AMDETXRX` | MDE cases | |
| Design | `VESTR_C`, `VEREP`, `ANALWT2_C1–C4` | all | needed for correct estimates |

`REQUIRED` concepts stop the pipeline if missing; the others are optional and are simply reported. If something optional isn't found, use `search_vars("...")` above to find its current name and add it to the candidate list.

In [ ]:
VARIABLE_CANDIDATES = {
    # concept          candidate names (first match wins)
    "resp_id":        ["QUESTID2", "QUESTID"],
    "year":           ["YEAR"],
    "age_cat":        ["AGE3"],
    "sex":            ["IRSEX"],
    "race_eth":       ["NEWRACE2"],
    "college_enroll": ["COLLENRLST"],
    # school variables are only a fallback if COLLENRLST is absent
    "school_now":     ["EDUSCHLGO"],
    "school_grade":   ["EDUSCHGRD2"],
    "school_ftpt":    ["EDUFULPAR"],
    # depression
    "ymde_lt":        ["YMDELT"],
    "ymde_py":        ["YMDEYR"],
    "amde_lt":        ["AMDELT"],
    "amde_py":        ["AMDEYR"],
    # psychological distress (adults)
    "k6_pm":          ["K6SCMON"],
    "k6_py_max":      ["K6SCMAX"],
    "spd_pm":         ["SPDMON", "SPDPSTMON"],
    "spd_py":         ["SPDYR", "SPDPSTYR"],
    # treatment
    "mh_tx_py":       ["MHTRTPY", "AMHTXRC4", "AMHTXRC3"],
    "ymde_tx":        ["YMDETXRX"],
    "amde_tx":        ["AMDETXRX"],
    # survey design
    "strata":         ["VESTR_C", "VESTR"],
    "psu":            ["VEREP"],
    "wt_1yr":         ["ANALWT2_C1", "ANALWT_C"],
    "wt_2yr":         ["ANALWT2_C2"],
    "wt_3yr":         ["ANALWT2_C3"],
    "wt_4yr":         ["ANALWT2_C4"],
}
REQUIRED = {"year", "age_cat", "sex", "race_eth",
            "ymde_py", "amde_py", "strata", "psu", "wt_1yr"}

SRC = {}          # concept -> NSDUH variable actually used
not_found = []
for concept, cands in VARIABLE_CANDIDATES.items():
    hit = next((c for c in cands if c in UPPER_TO_ORIG), None)
    (SRC.__setitem__(concept, hit) if hit else not_found.append(concept))

missing_required = REQUIRED & set(not_found)
if missing_required:
    raise KeyError(f"Required variables not found: {missing_required}. "
                   "Use search_vars() to locate their names in this release.")

print("Not found (optional):", not_found or "none")
pd.DataFrame([(k, v, VAR_LABELS.get(v, "")) for k, v in SRC.items()],
             columns=["concept", "nsduh_var", "label"])

## 3. Load only the selected columns

**What:** re-read the `.dta` with `usecols=` so only ~25 columns are loaded.

**Why:** this is the "trim the large file" step from the proposal's roadblocks. It cuts memory from gigabytes to a few dozen MB and makes every later step fast.

We also keep an untouched copy (`raw`) before any recoding. Some skip codes carry real information — e.g. a *legitimate skip* on a school-grade question means "not in school" — so derived variables that need those codes are built from `raw`.

In [ ]:
usecols = [UPPER_TO_ORIG[v] for v in SRC.values()]
df, _ = pyreadstat.read_dta(str(DTA_PATH), usecols=usecols)
df.columns = df.columns.str.upper()

if "resp_id" not in SRC:                      # build a stable ID if none exists
    df.insert(0, "RESP_ID", np.arange(1, len(df) + 1))
    SRC["resp_id"] = "RESP_ID"

raw = df.copy()
print(df.shape, f"— {df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
df.head()

## 4. Inspect each variable's codes

**What:** print the label, value labels, and observed values for every kept variable.

**Why:** before recoding anything, we confirm what each code means *in this file*. This catches surprises like recode variables using `1 = Yes / 2 = No` while others use `1 = Yes / 0 = No`, and shows which special codes (85, 94, 97, 98, 99, -9) actually appear.

In [ ]:
DESIGN_VARS = {SRC[k] for k in ("strata", "psu", "wt_1yr", "wt_2yr", "wt_3yr", "wt_4yr") if k in SRC}

for concept, var in SRC.items():
    if var in DESIGN_VARS or var == SRC["resp_id"]:
        continue
    vc = raw[var].value_counts(dropna=False).sort_index()
    print(f"\n=== {var}  ({concept}) — {VAR_LABELS.get(var, '')}")
    if value_labels(var):
        print("value labels:", value_labels(var))
    print(vc.head(30).to_string() if len(vc) <= 30 else
          f"{len(vc)} distinct values; min={raw[var].min()}, max={raw[var].max()}")

## 5. Recode missing-value codes

**What:** replace NSDUH's numeric "special" codes with real missing values (`NaN`).

NSDUH uses numbers for non-answers (from the codebook):

| Code | Meaning | Treat as missing? |
|---|---|---|
| 85 | Bad data (logically assigned) | yes |
| 94 | Don't know | yes |
| 97 | Refused | yes |
| 98 | Blank / no answer | yes |
| 99 | Legitimate skip (question not asked) | usually — but it can carry meaning |
| 81, 91 | Never used (substance items) | **no** — a real answer |
| 83, 93 | Did not use in period (substance items) | **no** — a real answer |
| -9 | Variable **not collected in that year** (concatenated file only) | yes |

**Why not just replace 85–99 everywhere?** Because the cutoff differs by variable. Continuous items (ages, day counts) use 985/9985 codes, some items start at 94, and on others 91/93 are valid answers. Replacing blindly would delete real data or leave codes like 994 in place and wreck averages.

**How:** SAMHSA ships the exact per-variable cutoff in the Stata syntax file (`replace X = . if (X >= cutoff)`). We **parse that file** into a rule dictionary, so our recoding is identical to SAMHSA's and stays correct when we add variables.

⚠️ **Bug in the `.do` file:** about 600 lines read `replace X = . if (X >= -9)`. Taken literally, that sets *every* value to missing (all valid codes are ≥ -9). The intent is clearly "-9 means not collected this year", so we treat those as `X == -9`.

Recoded/imputed variables (e.g. `AMDEYR`, `IRSEX`, `NEWRACE2`, K6 scores) have no user-defined missing codes and aren't in the `.do` file; for them we only convert negative codes (-9). Design variables are never touched.

In [ ]:
def parse_do_missing_rules(do_path):
    # Return {VARIABLE: cutoff} from lines like:  replace X = . if (X >= 85 )
    text = Path(do_path).read_text(errors="ignore")
    pat = re.compile(r"replace\s+(\w+)\s*=\s*\.\s*if\s*\(\s*\1\s*>=\s*(-?\d+)\s*\)", re.I)
    return {m.group(1).upper(): int(m.group(2)) for m in pat.finditer(text)}

MISSING_RULES = parse_do_missing_rules(DO_PATH)
n_neg9 = sum(v == -9 for v in MISSING_RULES.values())
print(f"Parsed {len(MISSING_RULES):,} rules from the .do file ({n_neg9} use the '>= -9' form)")

In [ ]:
def recode_missing(frame, rules, skip=()):
    # Apply SAMHSA missing-value rules; return (clean frame, log table).
    out, log = frame.copy(), []
    for col in out.columns:
        if col in skip:
            continue
        cutoff = rules.get(col)
        s = out[col]
        if cutoff is None or cutoff < 0:
            mask, rule = s < 0, "negative codes (-9) only"
        else:
            mask, rule = (s >= cutoff) | (s < 0), f">= {cutoff}  (+ -9)"
        log.append({"variable": col, "rule": rule,
                    "n_recoded": int(mask.sum()),
                    "pct_missing_after": round(100 * (s.isna() | mask).mean(), 2)})
        out.loc[mask, col] = np.nan
    return out, pd.DataFrame(log)

df, missing_log = recode_missing(df, MISSING_RULES, skip=DESIGN_VARS | {SRC["resp_id"]})
missing_log

**Missingness by year.** A variable that is ~100% missing in one year wasn't asked that year — important because the whole project compares years. Expect high missingness for youth items among adults (and vice versa); that's by design, not a data problem.

In [ ]:
check_vars = [v for k, v in SRC.items() if v not in DESIGN_VARS | {SRC["resp_id"], SRC["year"]}]
(df[check_vars].isna().groupby(df[SRC["year"]]).mean() * 100).round(1).T

## 6. Derive analysis variables

### 6a. Age group, sex, race/ethnicity

**Age group** comes from `AGE3` (1–3 = 12–17, 4–6 = 18–25, 7+ = 26+). We keep 26+ in the file (see Step 7 for why) but label it.

**Race/ethnicity:** `race_eth` keeps all 7 NSDUH categories. `race_eth5` collapses the three smallest groups (American Indian/Alaska Native, Native Hawaiian/Pacific Islander, multiracial) into *Other / Multiracial*. **Why:** the proposal flags that 18–22 college students broken down by race may have too few respondents per year; the collapsed version gives more stable yearly estimates, while the 7-group version is still available for pooled (multi-year) analysis.

All labelled variables are ordered categoricals so tables and plots sort sensibly.

In [ ]:
def ordered_cat(series, mapping):
    return pd.Categorical(series.map(mapping), categories=list(dict.fromkeys(mapping.values())), ordered=True)

AGE_GROUP = {1: "12-17", 2: "12-17", 3: "12-17",
             4: "18-25", 5: "18-25", 6: "18-25",
             7: "26+", 8: "26+", 9: "26+", 10: "26+", 11: "26+"}
SEX = {1: "Male", 2: "Female"}
RACE7 = {1: "White", 2: "Black", 3: "AIAN", 4: "NHPI", 5: "Asian",
         6: "Multiracial", 7: "Hispanic"}
RACE5 = {1: "White", 2: "Black", 7: "Hispanic", 5: "Asian",
         3: "Other/Multiracial", 4: "Other/Multiracial", 6: "Other/Multiracial"}

print("AGE3 labels:", value_labels(SRC["age_cat"]))
print("IRSEX labels:", value_labels(SRC["sex"]))
print("NEWRACE2 labels:", value_labels(SRC["race_eth"]))

clean = pd.DataFrame({
    "resp_id":   df[SRC["resp_id"]],
    "year":      df[SRC["year"]].astype("int16"),
    "age_cat":   df[SRC["age_cat"]].astype("Int8"),
    "age_group": ordered_cat(df[SRC["age_cat"]], AGE_GROUP),
    "sex":       ordered_cat(df[SRC["sex"]], SEX),
    "race_eth":  ordered_cat(df[SRC["race_eth"]], RACE7),
    "race_eth5": ordered_cat(df[SRC["race_eth"]], RACE5),
})
clean[["age_group", "sex", "race_eth5"]].apply(lambda s: s.value_counts(dropna=False)).T

### 6b. College enrollment (ages 18–22)

**What:** `college_status` = Full-time college / Part-time college / Not enrolled, and `college_ft_vs_other` = Full-time student vs everyone else aged 18–22.

**Why `COLLENRLST`:** the public file only gives age in bands (`AGE3` 4 = 18–20, 5 = 21–23), so we *cannot* isolate exactly 18–22 ourselves. SAMHSA's `COLLENRLST` recode is built from exact age and is only defined for 18–22, so it's the correct source.

**How the codes are mapped (no guessing):**
- A code is "age-ineligible" if it appears among 12–17-year-olds or adults 26+ — those people can't be 18–22. Done from the data, not from label text.
- Among eligible codes, the value label decides Full-time vs Part-time; everything else is "Not enrolled".
- The mapping table is printed for you to confirm, and an assertion checks that everyone placed in the 18–22 group is in `AGE3` 4 or 5.

**Comparison group:** the proposal compares full-time students to non-students. SAMHSA's college reports compare *full-time students* to *all other 18–22-year-olds* (part-time + not enrolled). We provide both so you can choose; just state which one you used.

In [ ]:
def build_college_status(raw, age_cat):
    var = SRC.get("college_enroll")
    if var is None:
        raise KeyError("COLLENRLST not found — see the fallback note below this cell.")
    codes = raw[var]
    labels = value_labels(var)
    rows = []
    for code_ in sorted(codes.dropna().unique()):
        ages = age_cat[codes == code_]
        ineligible = ages.isin([1, 2, 3]).any() or (ages >= 7).any()
        lbl = labels.get(code_, "")
        if ineligible:
            status = None
        elif re.search(r"full", lbl, re.I):
            status = "Full-time college"
        elif re.search(r"part", lbl, re.I):
            status = "Part-time college"
        else:
            status = "Not enrolled"
        rows.append({"code": code_, "label": lbl, "n": int((codes == code_).sum()),
                     "AGE3 range": f"{ages.min()}–{ages.max()}", "assigned": status})
    mapping = pd.DataFrame(rows)
    status = codes.map(dict(zip(mapping["code"], mapping["assigned"])))
    return status, mapping

college_status, college_map = build_college_status(raw, raw[SRC["age_cat"]])
display(college_map)

clean["college_status"] = pd.Categorical(
    college_status, categories=["Full-time college", "Part-time college", "Not enrolled"], ordered=True)
clean["college_ft_vs_other"] = pd.Categorical(
    np.where(clean["college_status"].isna(), None,
             np.where(clean["college_status"] == "Full-time college", "Full-time student", "Other 18-22")),
    categories=["Full-time student", "Other 18-22"])
clean["age_18_22"] = clean["college_status"].notna()

# Validation: everyone with a college status must be 18–23 by AGE3
assert clean.loc[clean["age_18_22"], "age_cat"].isin([4, 5]).all(), "college status assigned outside ages 18–23"
assert {"Full-time college"} <= set(college_map["assigned"].dropna()), "no full-time code found — check labels"
pd.crosstab(clean["year"], clean["college_status"], margins=True)

> **Fallback if `COLLENRLST` is not in your file:** derive from the raw school items — currently attending (`EDUSCHLGO`), grade = college/university (`EDUSCHGRD2`), full- vs part-time (`EDUFULPAR`) — restricted to `AGE3` 4–5. Use `raw`, not `df`, because a *legitimate skip* (99) on the grade question means "not in school". The limitation: `AGE3` 5 includes 23-year-olds, so the group would be 18–23, which you'd need to state in the report.

### 6c. Depression, distress, and treatment → clean 0/1 indicators

**What:** convert outcome variables to `1 = yes, 0 = no, NaN = missing/not applicable`.

**Why:** NSDUH recodes use `1 = Yes, 2 = No`. With 0/1, a (weighted) mean *is* the prevalence, which makes every later table one line of code. The helper checks the observed values first and refuses to guess if the coding is unexpected.

**Unified MDE:** `mde_py` / `mde_lt` take the **youth** measure for 12–17 and the **adult** measure for 18+, so one column tracks depression across both groups. We also keep the separate youth/adult columns.

⚠️ **For the report:** youth and adult MDE come from different questionnaire modules (wording adapted for adolescents), and SAMHSA advises caution comparing them directly. Comparing *trends* within each group (e.g. "did 12–17 rise more than 18–25?") is fine; comparing levels across groups should come with that caveat.

**K6 / SPD** are only asked of adults, so they stay missing for 12–17 by design — they can't be used for the adolescent comparison.

In [ ]:
def to_binary(s, name):
    vals = set(s.dropna().unique())
    if vals <= {0, 1}:
        return s.astype("Int8")
    if vals <= {1, 2}:
        return s.map({1: 1, 2: 0}).astype("Int8")
    raise ValueError(f"{name}: unexpected values {sorted(vals)[:10]} — check value labels before recoding")

def get(concept):
    return df[SRC[concept]] if concept in SRC else pd.Series(np.nan, index=df.index)

is_youth = clean["age_group"] == "12-17"
is_adult = clean["age_group"].isin(["18-25", "26+"])

for c in ["ymde_lt", "ymde_py", "amde_lt", "amde_py", "spd_pm", "spd_py", "mh_tx_py", "ymde_tx", "amde_tx"]:
    if c in SRC:
        clean[c] = to_binary(get(c), SRC[c])

# Unified depression measures (youth module for 12–17, adult module for 18+)
def col(c):
    return clean[c] if c in clean else pd.Series(pd.NA, index=clean.index, dtype="Int8")

clean["mde_py"] = col("ymde_py").where(is_youth, col("amde_py")).where(is_youth | is_adult)
clean["mde_lt"] = col("ymde_lt").where(is_youth, col("amde_lt")).where(is_youth | is_adult)
if "ymde_tx" in clean or "amde_tx" in clean:
    clean["mde_tx"] = col("ymde_tx").where(is_youth, col("amde_tx"))

# K6 scores stay numeric (0–24); adults only
for c in ["k6_pm", "k6_py_max"]:
    if c in SRC:
        clean[c] = get(c).where(is_adult)

# Sanity: youth MDE should only exist for youths, adult MDE only for adults
pd.DataFrame({
    "ymde_py non-missing": clean.groupby("age_group", observed=True)["ymde_py"].count(),
    "amde_py non-missing": clean.groupby("age_group", observed=True)["amde_py"].count(),
})

### 6d. Survey design variables

**What:** keep strata (`VESTR_C`), replicate/PSU (`VEREP`), and all four weight versions.

**Why:** NSDUH oversamples some groups, so unweighted percentages are **not** national estimates. The concatenated file provides weights pre-divided for pooling:

| Weight | Use when… |
|---|---|
| `wt_1yr` (`ANALWT2_C1`) | estimating **one year at a time** — the main year-by-year trend |
| `wt_2yr` / `wt_3yr` | pooling 2 or 3 years |
| `wt_4yr` (`ANALWT2_C4`) | pooling **all four years**, e.g. small groups like Asian full-time students |

Strata + PSU are needed for correct standard errors / confidence intervals (Taylor linearization), e.g. with Python's `samplics` or R's `survey` package. Point estimates only need the weight.

In [ ]:
for k in ["strata", "psu", "wt_1yr", "wt_2yr", "wt_3yr", "wt_4yr"]:
    if k in SRC:
        clean[k] = df[SRC[k]]

# The pooled weights should be exact fractions of the single-year weight
for k, d in [("wt_2yr", 2), ("wt_3yr", 3), ("wt_4yr", 4)]:
    if k in clean:
        assert np.allclose(clean[k], clean["wt_1yr"] / d, rtol=1e-6), f"{k} is not wt_1yr/{d}"
assert (clean["wt_1yr"] > 0).all(), "non-positive weights found"
print("weight checks passed")

## 7. Define the study population (flag, don't drop)

**What:** add `in_study` = ages 12–25, instead of deleting everyone 26+.

**Why:** in survey analysis, a subgroup should be analysed as a *domain* of the full sample. Dropping rows can remove whole strata/PSUs and make standard errors wrong. The column-trimmed file is small (~30 MB), so keeping all rows costs nothing. For quick EDA we also save a filtered 12–25 copy, but use the full file + `in_study` when computing confidence intervals.

In [ ]:
clean["in_study"] = clean["age_group"].isin(["12-17", "18-25"])
print(clean["in_study"].value_counts())
pd.crosstab(clean["year"], clean["age_group"], margins=True)

## 8. Validate

**What:** automated checks that fail loudly if something upstream is off, plus a quick look at weighted prevalence.

**Why:** a pipeline is only "automated" if it catches its own errors. The checks confirm the four years are present with plausible sizes, every indicator is 0/1/NaN, MDE only exists in the right age group, and K6 is in range.

In [ ]:
binary_cols = [c for c in clean.columns
               if c in {"ymde_lt", "ymde_py", "amde_lt", "amde_py", "mde_lt", "mde_py",
                        "spd_pm", "spd_py", "mh_tx_py", "ymde_tx", "amde_tx", "mde_tx"}]

checks = {
    "all four years present":            set(clean["year"]) == set(STUDY_YEARS),
    "each year has 40k–80k respondents": clean["year"].value_counts().between(40_000, 80_000).all(),
    "resp_id unique within year":        not clean.duplicated(["year", "resp_id"]).any(),
    "no missing age/sex/race":           clean[["age_group", "sex", "race_eth"]].notna().all().all(),
    "indicators are 0/1/NaN":            all(set(clean[c].dropna().unique()) <= {0, 1} for c in binary_cols),
    "youth MDE only for 12-17":          clean.loc[~is_youth, "ymde_py"].isna().all(),
    "adult MDE only for 18+":            clean.loc[is_youth, "amde_py"].isna().all(),
    "K6 within 0–24":                    ("k6_pm" not in clean) or clean["k6_pm"].dropna().between(0, 24).all(),
    "college status only ages 18-23":    clean.loc[clean["age_18_22"], "age_cat"].isin([4, 5]).all(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "Validation failed — see FAIL lines above"

**Weighted prevalence sanity check.** These should be close to SAMHSA's published NSDUH annual detailed tables for past-year MDE by age group — if they are wildly off, a recode is wrong. (Point estimates only; no standard errors here.) The unweighted `n` column flags small cells — SAMHSA suppresses imprecise estimates, and the proposal already anticipates pooling years for small 18–22 subgroups.

In [ ]:
def weighted_prevalence(data, outcome, by, weight="wt_1yr"):
    d = data.dropna(subset=[outcome]).copy()
    d["_wy"] = d[outcome].astype(float) * d[weight]
    g = d.groupby(by, observed=True)
    return pd.DataFrame({"pct": 100 * g["_wy"].sum() / g[weight].sum(),
                         "n_unweighted": g.size()}).round(1)

study = clean[clean["in_study"]]
weighted_prevalence(study, "mde_py", ["year", "age_group"]).unstack("age_group")

In [ ]:
# Same check for the college comparison (ages 18–22)
weighted_prevalence(study[study["age_18_22"]], "mde_py", ["year", "college_ft_vs_other"]).unstack()

## 9. Save outputs

**What:**
- `nsduh_2124_clean.parquet` — all respondents, cleaned (use this with survey-design methods)
- `nsduh_2124_study_12_25.parquet` — ages 12–25 only, for quick EDA / plotting
- `data_dictionary.csv` — every column, its source variable, and its meaning

**Why Parquet:** it preserves data types (ordered categories, nullable integers), is far smaller than CSV, and loads in a fraction of a second, so downstream notebooks don't redo any cleaning. A CSV is also written for anyone without `pyarrow`.

In [ ]:
DESCRIPTIONS = {
    "resp_id": "Respondent ID", "year": "Survey year",
    "age_cat": "AGE3 code (raw)", "age_group": "12-17 / 18-25 / 26+",
    "sex": "Male / Female", "race_eth": "Race/ethnicity, 7 NSDUH groups",
    "race_eth5": "Race/ethnicity, 5 groups (AIAN, NHPI, multiracial combined)",
    "college_status": "Ages 18-22: full-time / part-time / not enrolled",
    "college_ft_vs_other": "Ages 18-22: full-time student vs all other 18-22",
    "age_18_22": "True if respondent is 18-22 (has a college status)",
    "ymde_lt": "Youth lifetime MDE (12-17), 1/0", "ymde_py": "Youth past-year MDE (12-17), 1/0",
    "amde_lt": "Adult lifetime MDE (18+), 1/0", "amde_py": "Adult past-year MDE (18+), 1/0",
    "mde_lt": "Lifetime MDE, youth measure for 12-17 and adult for 18+",
    "mde_py": "Past-year MDE, youth measure for 12-17 and adult for 18+",
    "k6_pm": "K6 distress score past month, 0-24 (18+)",
    "k6_py_max": "K6 distress score, worst month past year, 0-24 (18+)",
    "spd_pm": "Serious psychological distress past month, 1/0 (18+)",
    "spd_py": "Serious psychological distress past year, 1/0 (18+)",
    "mh_tx_py": "Received mental health treatment past year, 1/0",
    "ymde_tx": "Youth with MDE: received treatment for depression, 1/0",
    "amde_tx": "Adult with MDE: received treatment for depression, 1/0",
    "mde_tx": "Received treatment for depression (if MDE), unified",
    "strata": "Variance stratum", "psu": "Variance replicate (PSU)",
    "wt_1yr": "Analysis weight, single-year estimates", "wt_2yr": "Weight for pooling 2 years",
    "wt_3yr": "Weight for pooling 3 years", "wt_4yr": "Weight for pooling all 4 years",
    "in_study": "True if age 12-25 (study population)",
}
dictionary = pd.DataFrame({
    "column": clean.columns,
    "source_variable": [SRC.get(c, "derived") for c in clean.columns],
    "description": [DESCRIPTIONS.get(c, "") for c in clean.columns],
    "dtype": clean.dtypes.astype(str).values,
    "pct_missing": (clean.isna().mean() * 100).round(1).values,
})

clean.to_parquet(OUT_DIR / "nsduh_2124_clean.parquet", index=False)
clean[clean["in_study"]].to_parquet(OUT_DIR / "nsduh_2124_study_12_25.parquet", index=False)
clean.to_csv(OUT_DIR / "nsduh_2124_clean.csv", index=False)
dictionary.to_csv(OUT_DIR / "data_dictionary.csv", index=False)

print(f"Saved {clean.shape[0]:,} rows x {clean.shape[1]} columns to {OUT_DIR}/")
dictionary

## Summary of cleaning decisions (for the report)

1. **Trimmed 2,638 → ~30 columns** by reading metadata first and loading only needed variables.
2. **Missing-value codes recoded with SAMHSA's own per-variable rules**, parsed from the official Stata syntax file; the `>= -9` lines in that file were corrected to `== -9` (not collected that year).
3. **College enrollment** taken from SAMHSA's `COLLENRLST`, since the public file has no exact age to isolate 18–22; codes mapped from the data and checked against age.
4. **Outcomes standardized to 0/1** so weighted means equal prevalences; youth and adult MDE combined into one column but kept separately too, with a caveat about comparing levels across modules.
5. **Race/ethnicity offered at 7 and 5 levels** to manage small cells; `wt_4yr` available for pooled estimates of small groups.
6. **No respondents dropped**; study population flagged with `in_study` so survey standard errors stay valid.
7. **Automated validation checks** stop the pipeline if any assumption breaks.

**Next steps:** compute year-by-group prevalences with confidence intervals using `strata`, `psu`, and `wt_1yr` (e.g. `samplics`), and compare against SAMHSA's published tables before building visualizations.